# Quels états le modèle et Adam doivent-ils conserver ?

Distinguer les couches enregistrées, les références fixes et l’historique de l’optimiseur.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#modules-et-buffers).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Après une sauvegarde simple, deux besoins se présentent : garder une référence de centrage avec le modèle et reprendre Adam sans oublier son passé. Ils concernent des états différents.

Supposons une moyenne fixe [0,1 ; 1,5], choisie pour la démonstration. Elle doit suivre le modèle lors d’un déplacement et d’une sauvegarde, sans être ajustée par l’optimiseur.

ModuleList enregistre les sous-couches ; une liste Python ordinaire ne le ferait pas. Elle ne définit pas leur enchaînement : forward les parcourt ici explicitement. register_buffer enregistre la moyenne comme état du modèle.

| État | parameters() | state_dict() |
| --- | --- | --- |
| Poids de la couche enregistrée | Oui | Oui |
| Moyenne : buffer persistant | Non | Oui |
| Moments et compteurs Adam | Non | Dans optimizer.state_dict() |

Les états du modèle et ceux de l’optimiseur sont distincts. Les buffers persistants sont inclus dans le dictionnaire du modèle.

## Démonstration 1 · Inspecter les inventaires du module

Vérifier la différence entre paramètre, buffer et état sauvegardé.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Définir la règle avant de l’utiliser

Le constructeur déclare une collection de couches et un tenseur fixe.

La boucle donne ici l’ordre d’application de ModuleList.

In [4]:
class ProjectionCentree(nn.Module):
    def __init__(self):
        super().__init__()
        self.couches = nn.ModuleList([nn.Linear(2, 2, bias=False)])
        self.register_buffer("moyenne", torch.tensor([0.1, 1.5]))
    def forward(self, x):
        valeurs = x - self.moyenne
        for couche in self.couches:
            valeurs = couche(valeurs)
        return valeurs


### Calculer l’état suivant

Les poids identité laissent passer les valeurs centrées.

In [5]:
modele = ProjectionCentree()


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [6]:
with torch.no_grad():
    modele.couches[0].weight.copy_(torch.eye(2))


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [7]:
x = torch.tensor([[0.2, 4.5]])
sortie = modele(x)


In [8]:
observer('x', x)
observer('sortie', sortie)

x forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[0.20000000298023224, 4.5]]
sortie forme (1, 2) type torch.float32 requires_grad True
  valeurs : [[0.10000000149011612, 3.0]]


### Calculer l’état suivant

Le buffer apparaît dans la sauvegarde, mais pas parmi les paramètres.

In [9]:
noms_parametres = list(dict(modele.named_parameters()))
noms_buffers = list(dict(modele.named_buffers()))
noms_etat = list(modele.state_dict())


In [10]:
observer('noms_parametres', noms_parametres)
observer('noms_buffers', noms_buffers)
observer('noms_etat', noms_etat)

noms_parametres = ['couches.0.weight']
noms_buffers = ['moyenne']
noms_etat = ['moyenne', 'couches.0.weight']


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [11]:
print('sortie :', sortie.tolist())
print('parametres :', noms_parametres)
print('buffers :', noms_buffers)
print('etat :', noms_etat)

sortie : [[0.10000000149011612, 3.0]]
parametres : ['couches.0.weight']
buffers : ['moyenne']
etat : ['moyenne', 'couches.0.weight']


In [12]:
assert bool(noms_parametres == ['couches.0.weight'] and noms_buffers == ['moyenne']), "modules-et-buffers/main: noms_parametres == ['couches.0.weight'] and noms_buffers == ['moyenne']"
assert bool(set(noms_etat) == {'moyenne', 'couches.0.weight'}), "modules-et-buffers/main: set(noms_etat) == {'moyenne', 'couches.0.weight'}"
assert bool(torch.allclose(sortie, torch.tensor([[0.1, 3.]]))), 'modules-et-buffers/main: torch.allclose(sortie, torch.tensor([[0.1, 3.]]))'
assert bool(not modele.moyenne.requires_grad), 'modules-et-buffers/main: not modele.moyenne.requires_grad'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


Un calcul peut fonctionner avec des couches non enregistrées ; ces inventaires vérifient ce que le modèle retrouve vraiment.

### Reprendre les prochains pas d’Adam

Reprenons le classifieur connu. Le premier pas utilise seulement la voix A ; le suivant utilisera les trois exemples fictifs. Cette variation rend l’effet de la mémoire observable. Après le premier pas, Adam a déjà mémorisé des moyennes et un compteur pour chaque paramètre.

Le programme continue une fois l’original, la copie restaurée et une copie aux mêmes poids mais avec Adam neuf. La reprise complète retrouve exactement l’original ; Adam neuf donne un résultat différent. Ce test est déterministe, sans chargement aléatoire de données.

Une expérience réelle peut aussi demander les états des générateurs aléatoires, du scheduler, du scaler et la position dans les données. Le fichier ne sauvegarde ni la définition Python de l’architecture ni son mode train/eval.

## Démonstration 2 · Comparer une vraie continuation

Contrôler le pas suivant, pas seulement l’existence d’un fichier.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [13]:
import torch
from torch import nn


### Calculer l’état suivant

Les données et poids initiaux sont imposés ; aucune graine n’est nécessaire pour ces valeurs.

In [14]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)
modele = nn.Linear(2, 2)


In [15]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [16]:
with torch.no_grad():
    modele.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    modele.bias.zero_()


### Choisir les paramètres et leur mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [17]:
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.01)


### Définir la règle avant de l’utiliser

La fonction reçoit le lot explicitement. Ce premier appel crée les états d’Adam sur le premier exemple.

In [18]:
def un_pas(reseau, optim, x, cible):
    optim.zero_grad()
    scores = reseau(x)
    perte = nn.CrossEntropyLoss()(scores, cible)
    perte.backward()
    optim.step()


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [19]:
un_pas(modele, optimiseur, X[:1], cibles[:1])


### Calculer l’état suivant

Le fichier contient deux dictionnaires d’état et le repère du pas.

In [20]:
torch.save({"model": modele.state_dict(),
            "optim": optimiseur.state_dict(), "pas": 1}, "reprise-adam.pt")
etat = torch.load("reprise-adam.pt", map_location="cpu", weights_only=True)


In [21]:
observer('etat', etat)

etat : clés ['model', 'optim', 'pas']


### Calculer l’état suivant

Le nouvel optimiseur doit référencer les paramètres de la copie.

In [22]:
copie = nn.Linear(2, 2)
copie.load_state_dict(etat["model"])
optim_copie = torch.optim.Adam(copie.parameters(), lr=0.01)
optim_copie.load_state_dict(etat["optim"])


In [23]:
observer('copie', copie)
observer('optim_copie', optim_copie)

### Calculer l’état suivant

Le troisième modèle reprend seulement les poids. Tous traitent ensuite le même lot de trois exemples.

In [24]:
neuf = nn.Linear(2, 2)
neuf.load_state_dict(etat["model"])
optim_neuf = torch.optim.Adam(neuf.parameters(), lr=0.01)
un_pas(modele, optimiseur, X, cibles)


In [25]:
observer('neuf', neuf)
observer('optim_neuf', optim_neuf)

### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [26]:
un_pas(copie, optim_copie, X, cibles)
un_pas(neuf, optim_neuf, X, cibles)


### Calculer l’état suivant

L’égalité des paramètres teste la continuation. Le compteur restauré passe à deux.

In [27]:
reprise_identique = all(torch.equal(a, b) for a, b in zip(modele.parameters(), copie.parameters()))
ecart_sans_etat = (modele.weight - neuf.weight).abs().max().item()
compteur = optim_copie.state[copie.weight]["step"].item()


In [28]:
observer('reprise_identique', reprise_identique)
observer('ecart_sans_etat', ecart_sans_etat)
observer('compteur', compteur)

reprise_identique = True
ecart_sans_etat = 0.003066958859562874
compteur = 2.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [29]:
print('continuation identique :', reprise_identique)
print('compteur Adam restaure puis avance :', compteur)
print('ecart avec Adam neuf :', round(ecart_sans_etat, 6))

continuation identique : True
compteur Adam restaure puis avance : 2.0
ecart avec Adam neuf : 0.003067


In [30]:
assert bool(reprise_identique and compteur == 2), 'modules-et-buffers/reprise-adam: reprise_identique and compteur == 2'
assert bool(ecart_sans_etat > 1e-4), 'modules-et-buffers/reprise-adam: ecart_sans_etat > 1e-4'
assert bool(len(optim_copie.state) == 2 and etat["pas"] == 1), 'modules-et-buffers/reprise-adam: len(optim_copie.state) == 2 and etat["pas"] == 1'
assert bool(torch.equal(modele.bias, copie.bias)), 'modules-et-buffers/reprise-adam: torch.equal(modele.bias, copie.bias)'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


L’état de l’optimiseur fait partie de ce qui définit le prochain déplacement.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Un buffer moyenne est enregistré. Prévoyez sa présence dans state_dict, puis dans named_parameters, avec deux booléens.

In [31]:
exo_modele = torch.nn.Linear(2,2)
exo_modele.register_buffer("moyenne",torch.zeros(2))
exo_presences = ...  # Remplacer les points par votre réponse.
if exo_presences is Ellipsis or exo_presences is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_presences) == ("moyenne" in exo_modele.state_dict(),"moyenne" in dict(exo_modele.named_parameters()))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le buffer persistant est sauvegardé et déplacé avec le modèle, mais reste hors des paramètres ajustables.

</details>

### Exercice 2

Après un pas Adam, fournissez l’état à recharger dans le nouvel optimiseur. Les poids du modèle sont déjà copiés.

In [32]:
exo_modele = torch.nn.Linear(2,2)
exo_opt = torch.optim.Adam(exo_modele.parameters(),lr=.01)
exo_perte = exo_modele(torch.ones(1,2)).square().mean()
exo_perte.backward()
exo_opt.step()
exo_copie = torch.nn.Linear(2,2)
exo_copie.load_state_dict(exo_modele.state_dict())
exo_opt_copie = torch.optim.Adam(exo_copie.parameters(),lr=.01)
exo_etat_optim = ...  # Remplacer les points par votre réponse.
if exo_etat_optim is Ellipsis or exo_etat_optim is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_opt_copie.load_state_dict(exo_etat_optim)
    assert len(exo_opt_copie.state) == 2
    for exo_a,exo_b in zip(exo_opt.state.values(),exo_opt_copie.state.values()):
        assert exo_b["step"].item() == 1
        assert torch.equal(exo_a["exp_avg"],exo_b["exp_avg"])
        assert torch.equal(exo_a["exp_avg_sq"],exo_b["exp_avg_sq"])
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Les moments et compteurs appartiennent à optimizer.state_dict(), pas au dernier nombre de perte.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_modele = torch.nn.Linear(2,2)
exo_modele.register_buffer("moyenne",torch.zeros(2))
exo_presences = (True,False)
if exo_presences is Ellipsis or exo_presences is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_presences) == ("moyenne" in exo_modele.state_dict(),"moyenne" in dict(exo_modele.named_parameters()))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_modele = torch.nn.Linear(2,2)
exo_opt = torch.optim.Adam(exo_modele.parameters(),lr=.01)
exo_perte = exo_modele(torch.ones(1,2)).square().mean()
exo_perte.backward()
exo_opt.step()
exo_copie = torch.nn.Linear(2,2)
exo_copie.load_state_dict(exo_modele.state_dict())
exo_opt_copie = torch.optim.Adam(exo_copie.parameters(),lr=.01)
exo_etat_optim = exo_opt.state_dict()
if exo_etat_optim is Ellipsis or exo_etat_optim is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_opt_copie.load_state_dict(exo_etat_optim)
    assert len(exo_opt_copie.state) == 2
    for exo_a,exo_b in zip(exo_opt.state.values(),exo_opt_copie.state.values()):
        assert exo_b["step"].item() == 1
        assert torch.equal(exo_a["exp_avg"],exo_b["exp_avg"])
        assert torch.equal(exo_a["exp_avg_sq"],exo_b["exp_avg_sq"])
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [35]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()